# 04 — LoRA and PEFT

Learn how to adapt a large pretrained model while training only a small number of parameters.

> **Course principle:** every notebook separates **general-purpose knowledge** from **Manga Task application**.

## LoRA concept

For a frozen matrix \(W\), LoRA learns a low-rank update:

\[
W' = W + \Delta W,\qquad \Delta W = BA
\]

with rank \(r\) much smaller than the full matrix dimensions.

Main knobs: rank `r`, `lora_alpha`, dropout, and `target_modules`.

In [ ]:
from peft import LoraConfig

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
print(config)

## Trainable-parameter inspection

Never assume LoRA was applied correctly. Count trainable parameters.

In [ ]:
def trainable_stats(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {
        "trainable": trainable,
        "total": total,
        "pct_trainable": 100 * trainable / total
    }

# print(trainable_stats(model))

## Manga-task adaptation choices

Possible adaptations include:
- language-side LoRA for output format, filtering rules and task behavior;
- projector LoRA if image-language alignment needs adjustment;
- some vision-side modules if manga line art differs strongly from pretraining.

Start narrow. Add trainable modules only when error analysis justifies them.

## QLoRA mental model

QLoRA typically quantizes frozen base weights while training LoRA adapters in higher precision. It reduces memory use but adds implementation constraints.

Do not add quantization just because it is fashionable; first determine whether memory requires it.